# Econometric Credit Risk Toolbox Walkthrough

This notebook shows the natural baseline credit-risk pipeline first, then demonstrates optional econometric extensions. The extensions are tools to test assumptions and compare alternatives, not one combined maximal model.

Baseline pipeline:
1. multi-period retail loan portfolio generation,
2. pooled-logit survival-based PD estimation,
3. IFRS 9 stage 1 / 2 / 3 logic,
4. scenario-weighted ECL with interpretable LGD/EAD,
5. data-quality and drift monitoring,
6. split-friendly validation pack for backtesting and governance.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from credit_risk_lab import (
    PortfolioConfig,
    fit_survival_pd_model,
    generate_portfolio_timeseries,
    run_ifrs9_pipeline,
    run_monitoring,
    score_portfolio,
)
from credit_risk_lab.ecl import default_macro_scenarios
from credit_risk_lab.econometrics import (
    build_macro_scenario_paths,
    score_forward_pd_paths,
    fit_ar1_macro_model,
    forecast_macro_path,
    calibration_table,
    compare_binary_model_specs,
    CandidateSpec,
    segment_performance_table,
    fit_markov_transition_model,
    fit_covariate_transition_model,
    compare_markov_to_survival_pd,
    fit_macro_regime_transition_matrices,
    build_markov_scenario_matrices,
    matrix_log_generator,
    reversibility_diagnostics,
    score_smoothness_diagnostics,
    estimate_piecewise_default_intensity,
    build_default_counting_process,
    build_continuous_credit_state_grid,
    ou_credit_quality_generator,
    beurling_deny_credit_decomposition,
    default_probability_from_generator,
)


## 1. Baseline Credit Risk Pipeline

The baseline should be run before any heavier extension. It gives a transparent benchmark for later model comparison.


In [ ]:
dataset = generate_portfolio_timeseries(PortfolioConfig(random_seed=42, periods=12, num_borrowers=250, num_loans=350))
pd_model = fit_survival_pd_model(dataset.performance)
as_of_date = dataset.performance['snapshot_date'].max()
scores = score_portfolio(dataset, pd_model, as_of_date)
ecl = run_ifrs9_pipeline(dataset, scores, scenarios=default_macro_scenarios())
monitoring = run_monitoring(scores.previous_snapshot_scores, scores.snapshot_scores, scores.previous_snapshot_scores['pd_12m'], scores.snapshot_scores['pd_12m'])
scores.snapshot_scores.head(), ecl.stage_summary, monitoring.score_drift


## 2. Check Calibration And Model Selection

Use these diagnostics before adding complexity. If the baseline is calibrated and stable, a heavier model needs to prove it adds value.


In [ ]:
calibration = calibration_table(scores.scored_history, 'pd_12m', 'observed_default_12m', n_bins=6)
comparison = compare_binary_model_specs(
    dataset.performance,
    'default_next_period',
    [
        CandidateSpec('ltv_dti', ('ltv', 'dti')),
        CandidateSpec('behavioural', ('ltv', 'dti', 'days_past_due', 'rating_rank'), ('segment',)),
    ],
)
calibration, comparison


## 3. Test Whether A Non-Constant Forward Hazard Is Needed

A non-constant path is not automatically better. It should be used when projected loan features or macro paths materially change the hazard term structure.


In [ ]:
macro_model = fit_ar1_macro_model(dataset.macro, ['unemployment_rate', 'policy_rate', 'house_price_growth'])
baseline_macro = forecast_macro_path(macro_model, horizon_quarters=8)
macro_paths = build_macro_scenario_paths(baseline_macro, {'baseline': {}, 'downside': {'unemployment_rate': 0.015}})
forward_pd = score_forward_pd_paths(scores.snapshot_scores.head(20), pd_model, macro_paths['downside'], max_horizon_quarters=8)
forward_pd[['loan_id', 'pd_12m_forward', 'pd_lifetime_forward']].head()


## 4. Test Heterogeneity Before Splitting Models

Segment performance tables show whether a single pooled model is hiding material differences. If segment behaviour is stable, separate models may add noise.


In [ ]:
segment_performance_table(scores.scored_history, 'segment', 'pd_12m', 'observed_default_12m')


## 5. Markov Migration Challenger

The survival model estimates default hazard. The Markov model explains movement through delinquency states and gives a challenger default probability.


In [ ]:
markov = fit_markov_transition_model(dataset.performance, smoothing=0.1)
cov_markov = fit_covariate_transition_model(
    markov.transition_panel,
    feature_columns=['rating_rank', 'days_past_due', 'ltv', 'unemployment_rate'],
    categorical_columns=['segment'],
    min_rows=10,
)
markov_sample = markov.transition_panel.head(50).assign(pd_12m=0.05)
markov_vs_survival = compare_markov_to_survival_pd(markov_sample, 'pd_12m', 4, covariate_model=cov_markov)
regime_matrices = fit_macro_regime_transition_matrices(markov.transition_panel, smoothing=0.1)
scenario_matrices = build_markov_scenario_matrices(regime_matrices, markov.transition_matrix)
markov.transition_matrix, markov_vs_survival.head(), list(scenario_matrices)


## 6. Generator, Reversibility, And Score Smoothness

Use the matrix logarithm as a diagnostic. It is accepted only when the candidate has valid non-negative off-diagonal transition rates and zero row sums.


In [ ]:
log_candidate = matrix_log_generator(markov.transition_matrix)
state_scores = {'current': 1.0, 'dpd_1_29': 2.0, 'dpd_30_89': 3.0, 'default': 5.0, 'prepay_mature': 0.0}
reversibility = reversibility_diagnostics(markov.transition_matrix)
smoothness = score_smoothness_diagnostics(markov.transition_matrix, state_scores)
log_candidate.message, reversibility.head(), smoothness['top_edges']


## 7. Continuous-Time Counting Process

Use this when default or transition timing is observed more finely than a quarterly panel.


In [ ]:
portfolio_intensity = estimate_piecewise_default_intensity(dataset.performance)['intensity'].iloc[0]
counting = build_default_counting_process(dataset.performance, intensity=portfolio_intensity)
counting.tail()


## 8. Continuous-State Dirichlet Diagnostics

This extension compares gradual local movement, sudden jumps, and killing/default in a latent credit-quality process. It is a diagnostic process-design tool, not a default replacement for the observed quarterly model.


In [ ]:
grid = build_continuous_credit_state_grid(x_min=-3, x_max=3, n_points=51, default_boundary=-1.5)
generator = ou_credit_quality_generator(grid, volatility=0.35, killing_intensity=0.01, downward_jump_intensity=0.02)
start_state = generator.index[len(generator) // 2]
active_states = [state for state in generator.index if state != 'default']
values = {state: idx / max(len(active_states) - 1, 1) for idx, state in enumerate(active_states)}
default_path = default_probability_from_generator(generator, start_state, [0, 1, 2, 4])
energy = beurling_deny_credit_decomposition(generator, values)
default_path, energy
